# 第 11 课作业：把稀疏连接变成可顺序读取的数据

这份作业对应 **第 11 课：为什么不能每次 spike 都扫描所有突触？**

本题要把 edge list 变成 source_index + contiguous records。真正的学习目标不是“会写嵌套循环”，而是理解 **一个 source 怎样直接定位自己的连续 synapse range**。

## 数据 contract

输入 edge 是 (source, target, weight)。

输出：

- source_index[source] = (start, count)
- records 连续保存 (target, weight)

要求：

- source ID 范围是 0 到 num_sources-1；
- 同一 source 内保留输入 edge 的原始顺序；
- zero-fanout source 也必须有合法的 (start, 0)；
- lookup_source() 只返回该 source 的连续 slice。

## 先不用代码：手工 pack 一个小图

设 num_sources = 3，edge 顺序为：

- (0, 2, +5)
- (2, 0, -1)
- (2, 1, +4)

请先写：

1. records 应该是什么顺序？
2. source 0 的 (start, count) 是什么？
3. source 1 没有 outgoing edge，它的 (start, count) 应该怎样写？
4. source 2 的 range 从哪里开始、有几条？
5. 分别用 slice 反查 source 0、1、2，确认都得到正确连接。

如果手工结果还不稳定，先不要写 build_source_index()。

## 写代码前的实现规划

这题第一次同时碰到“数据布局”和“构建算法”，所以先把工作拆开，但不要写伪代码答案。

你需要确保四件事：

1. 每个 source 都能对应到自己的记录组，即使这个组为空；
2. 最终 records 按 source 顺序形成连续区间；
3. 每个 source 的 index 记录这个区间从哪里开始、长度是多少；
4. runtime lookup 只使用 (start, count)，不再扫描原始 edge list。

如果 build_source_index() 一上来就很难写，先在纸上画出“source 0 的组、source 1 的组、source 2 的组”，再考虑怎样把这些组依次拼成 records。

## Part A：构建 source index 与 records

### 这个函数做什么？

`build_source_index()` 把原始 edge list 转成两份适合后续按 source 顺序读取的数据：

1. `source_index`：告诉我们“某个 source 的 records 从哪里开始、有多少条”；
2. `records`：真正连续保存 target 和 weight 的列表。

### 输入

- `num_sources`：系统一共有多少个 source；合法 source ID 是 `0 ... num_sources-1`；
- `edges`：原始连接列表，每条 edge 是  
  `(source, target, weight)`。

### 输出

函数返回 **两个列表**，顺序固定为：

`(source_index, records)`

其中：

1. `source_index`：长度为 `num_sources`。  
   `source_index[source] = (start, count)`：
   - `start` 是这个 source 的第一条 record 在 `records` 中的起始位置；
   - `count` 是这个 source 一共有多少条 outgoing synapse。
2. `records`：连续保存的 `(target, weight)` 列表。  
   同一 source 的 records 必须相邻，并保留该 source 在输入 edge list 中原有的顺序。

即使某个 source 没有 outgoing synapse，也必须在 `source_index` 中留下一个 `(start, 0)` entry。

In [ ]:
Edge = tuple[int, int, int]   # (source, target, weight)
Record = tuple[int, int]      # (target, weight)


def build_source_index(
    num_sources: int,
    edges: list[Edge],
) -> tuple[list[tuple[int, int]], list[Record]]:
    # YOUR CODE STARTS HERE
    raise NotImplementedError("TODO: pack edges by source")
    # YOUR CODE ENDS HERE
    return source_index, records

## Part B：按 (start, count) 查一个 source

### 这个函数做什么？

`lookup_source()` 在 event-time 接收一个 `source_id`，直接通过已经构建好的 `source_index` 找到它在 `records` 中的连续区间。

它不应该重新扫描原始 edge list。

### 输入

- `source_id`：当前发生 spike 的 source；
- `source_index`：Part A 生成的 `(start, count)` 索引；
- `records`：Part A 生成的连续 `(target, weight)` records。

### 输出

函数返回 **一个 record 列表**：

- 只包含这个 `source_id` 对应的 `(target, weight)`；
- 顺序与 `records` 中保持一致；
- 如果该 source 的 `count = 0`，返回空列表 `[]`。

这里的返回值不是整个 records store，而只是当前 source 的那一个 slice。

In [ ]:
def lookup_source(
    source_id: int,
    source_index: list[tuple[int, int]],
    records: list[Record],
) -> list[Record]:
    # YOUR CODE STARTS HERE
    raise NotImplementedError("TODO: slice by start and count")
    # YOUR CODE ENDS HERE

## 检查你的实现

grader 会检查 exact ranges、zero fanout、lookup slice 和同一 source 内的 record order。

In [ ]:
# Course infrastructure: make the repository root importable from a notebook subdirectory.
from pathlib import Path
import sys

_repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "exercises" / "grader").is_dir()
)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from exercises.grader.lesson11 import check

check(
    build_source_index=build_source_index,
    lookup_source=lookup_source,
    language="zh",
)

## Human Check

用自己的实现做故障定位：

1. build_source_index() 属于 build-time preprocessing；lookup_source() 属于 event-time lookup。为什么后者绝不能重新扫描原始 edge list？
2. 如果某个 source 的 lookup 多拿了下一条邻居的 record，你会优先怀疑 start、count，还是 target accumulator？为什么？
3. zero-fanout source 的 count=0 时，start 仍然有什么意义？
4. 指着你的 lookup_source()：为什么它完全不需要知道每条 record 原来的 source_id？
5. 这套表示真正节省的是“每次 spike 要检查的记录数量”，还是保证所有 memory access 都一定更快？说明两者为什么不能混为一谈。